# Reward Function Lab: How Rewards Shape Allocator Behavior

**Goal:** Test 4 different reward designs side-by-side and see how they shape your allocator's behavior.

**The Key Insight:** Your reward function IS your trading strategy.

**What you'll learn:**
- Why raw returns train a trend-chasing machine
- How risk-adjusted rewards create conservative allocators
- When regret-relative rewards work best
- How to design custom rewards for your goals

**This is the most important notebook in the module.**

In [ ]:
callout("** Your reward function IS your trading strategy", kind="insight")

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries loaded successfully!")

In [ ]:
learning_objectives(["**Raw returns are dangerous** → Create trend-chasing machines that buy tops", "**Risk-adjusted rewards** → Conservative allocators that avoid volatility", "**Regret-relative rewards** → Balanced exploration and exploitation", "**Stability-weighted rewards** → Low turnover, tax/cost efficient"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Step 1: Load Commodity Data

Same 5 commodities as before.

In [ ]:
commodities = {
    'WTI': 'CL=F',
    'Gold': 'GC=F',
    'Copper': 'HG=F',
    'NatGas': 'NG=F',
    'Corn': 'ZC=F'
}

try:
    data = yf.download(
        list(commodities.values()),
        start='2021-01-01',
        end='2024-01-01',
        progress=False
    )['Adj Close']
    data.columns = list(commodities.keys())
except:
    print("Using synthetic data...")
    dates = pd.date_range('2021-01-01', '2024-01-01', freq='D')
    np.random.seed(42)
    data = pd.DataFrame({
        'WTI': 70 + np.cumsum(np.random.randn(len(dates)) * 2),
        'Gold': 1800 + np.cumsum(np.random.randn(len(dates)) * 10),
        'Copper': 4.0 + np.cumsum(np.random.randn(len(dates)) * 0.05),
        'NatGas': 3.0 + np.cumsum(np.random.randn(len(dates)) * 0.3),
        'Corn': 550 + np.cumsum(np.random.randn(len(dates)) * 5)
    }, index=dates)

weekly_prices = data.resample('W').last()
weekly_returns = weekly_prices.pct_change().dropna()

print(f"Loaded {len(weekly_returns)} weeks of returns")
weekly_returns.head()

## Step 2: Four Different Reward Functions

We'll test these reward designs:
1. **Raw Returns** (BAD): Maximize last week's return
2. **Risk-Adjusted** (GOOD): Sharpe-like with drawdown penalty
3. **Regret-Relative** (GOOD): Beat the best arm in hindsight
4. **Stability-Weighted** (GOOD): Penalize portfolio turnover

In [ ]:
class RewardFunctions:
    """Collection of reward function designs."""
    
    @staticmethod
    def raw_returns(arm_returns, **kwargs):
        """BAD: Just maximize last period's return."""
        return arm_returns  # This leads to trend-chasing
    
    @staticmethod
    def risk_adjusted(arm_returns, volatility=None, **kwargs):
        """GOOD: Sharpe-like, penalize volatility."""
        if volatility is None:
            volatility = np.abs(arm_returns) * 2  # Rough proxy
        return arm_returns / (volatility + 1e-6)
    
    @staticmethod
    def regret_relative(arm_returns, all_returns, **kwargs):
        """GOOD: Minimize regret vs best possible."""
        best_return = all_returns.max()
        regret = best_return - arm_returns
        return -regret  # Minimize regret
    
    @staticmethod
    def stability_weighted(
        arm_returns, 
        prev_weights, 
        curr_weights, 
        lambda_churn=1.0,
        **kwargs
    ):
        """GOOD: Penalize excessive rebalancing."""
        if prev_weights is None:
            return arm_returns
        
        # Turnover penalty
        turnover = np.abs(curr_weights - prev_weights).sum()
        portfolio_return = np.dot(curr_weights, arm_returns)
        
        return portfolio_return - lambda_churn * turnover

print("Reward functions defined:")
print("  1. Raw Returns (trend-chasing)")
print("  2. Risk-Adjusted (conservative)")
print("  3. Regret-Relative (balanced)")
print("  4. Stability-Weighted (low-turnover)")

## Step 3: Bandit with Configurable Rewards

Same Thompson Sampling, but reward function is pluggable.

In [ ]:
class ConfigurableBandit:
    """
    Thompson Sampling bandit with pluggable reward function.
    """
    def __init__(self, K, reward_func, prior_mean=0.001, prior_std=0.02):
        self.K = K
        self.reward_func = reward_func
        self.means = np.full(K, prior_mean)
        self.stds = np.full(K, prior_std)
        self.n = np.zeros(K)
        self.prev_weights = None
    
    def get_allocation(self):
        """Thompson Sampling allocation."""
        samples = np.random.normal(self.means, self.stds)
        exp_samples = np.exp(samples - samples.max())
        weights = exp_samples / exp_samples.sum()
        return weights
    
    def update(self, returns_array, **reward_kwargs):
        """
        Update beliefs using configured reward function.
        """
        curr_weights = self.get_allocation()
        
        # Compute rewards for each arm
        rewards = np.zeros(self.K)
        for i in range(self.K):
            rewards[i] = self.reward_func(
                arm_returns=returns_array[i],
                all_returns=returns_array,
                prev_weights=self.prev_weights,
                curr_weights=curr_weights,
                **reward_kwargs
            )
        
        # Bayesian update
        for i in range(self.K):
            self.n[i] += 1
            lr = 1 / (self.n[i] + 1)
            self.means[i] = (1 - lr) * self.means[i] + lr * rewards[i]
            self.stds[i] = self.stds[i] / np.sqrt(1 + self.n[i])
        
        self.prev_weights = curr_weights

print("Configurable bandit ready!")

## Step 4: Run All Four Bandits in Parallel

Same data, same environment, four different reward functions.

In [ ]:
def backtest_with_reward(returns_df, reward_func, name):
    """
    Backtest bandit with specific reward function.
    """
    K = len(returns_df.columns)
    bandit = ConfigurableBandit(K, reward_func)
    
    results = []
    
    for date, row in returns_df.iterrows():
        weights = bandit.get_allocation()
        returns_array = row.values
        portfolio_return = np.dot(weights, returns_array)
        
        results.append({
            'date': date,
            'return': portfolio_return,
            'turnover': 0 if bandit.prev_weights is None else 
                       np.abs(weights - bandit.prev_weights).sum(),
            **{f'weight_{col}': w for col, w in zip(returns_df.columns, weights)}
        })
        
        # Compute volatility for risk-adjusted reward
        volatility = returns_df.rolling(4).std().loc[date].values
        bandit.update(returns_array, volatility=volatility)
    
    df = pd.DataFrame(results).set_index('date')
    df['cumulative'] = (1 + df['return']).cumprod()
    return df

# Run all four
print("Running backtests with different reward functions...\n")

strategies = {
    'Raw Returns': RewardFunctions.raw_returns,
    'Risk-Adjusted': RewardFunctions.risk_adjusted,
    'Regret-Relative': RewardFunctions.regret_relative,
    'Stability-Weighted': RewardFunctions.stability_weighted
}

results_all = {}
for name, reward_func in strategies.items():
    print(f"  Running {name}...")
    results_all[name] = backtest_with_reward(weekly_returns, reward_func, name)

print("\nBacktests complete!")

## Step 5: Compare Cumulative Returns

How did each reward function perform?

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#E63946', '#2A9D8F', '#F4A261', '#264653']
linestyles = ['-', '--', '-.', ':']

for (name, result), color, ls in zip(results_all.items(), colors, linestyles):
    ax.plot(
        result.index,
        result['cumulative'],
        label=name,
        linewidth=2.5,
        color=color,
        linestyle=ls
    )

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return (Base = 1.0)', fontsize=12)
ax.set_title(
    'How Different Reward Functions Shape Performance',
    fontsize=14,
    fontweight='bold'
)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNotice the different equity curves!")
print("Each reward function creates a different trading strategy.")

## Step 6: Compare Allocation Patterns

How did each reward function allocate capital?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, result) in enumerate(results_all.items()):
    ax = axes[idx]
    
    # Extract weights
    weight_cols = [col for col in result.columns if col.startswith('weight_')]
    weights = result[weight_cols]
    weights.columns = [col.replace('weight_', '') for col in weights.columns]
    
    # Plot stacked area
    ax.stackplot(
        weights.index,
        *[weights[col] for col in weights.columns],
        labels=weights.columns,
        alpha=0.8
    )
    
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Allocation', fontsize=10)
    ax.set_ylim([0, 1])
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  - Raw Returns: Frequent, dramatic shifts (chasing trends)")
print("  - Risk-Adjusted: Smoother, more conservative")
print("  - Regret-Relative: Balanced exploration and exploitation")
print("  - Stability-Weighted: Minimal changes (low turnover)")

## Step 7: Compare Portfolio Turnover

How much trading did each strategy require?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

turnover_data = {}
for name, result in results_all.items():
    turnover_data[name] = result['turnover'].mean()

bars = ax.bar(
    turnover_data.keys(),
    turnover_data.values(),
    color=['#E63946', '#2A9D8F', '#F4A261', '#264653'],
    alpha=0.8
)

ax.set_ylabel('Average Weekly Turnover', fontsize=12)
ax.set_title('Portfolio Turnover by Reward Function', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar, value in zip(bars, turnover_data.values()):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{value:.3f}',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

print("\nTurnover matters for transaction costs!")
print("At 5 bps per trade, high turnover can eat 1-2% annual returns.")

## Step 8: Performance Metrics Table

Comprehensive comparison across all metrics.

In [ ]:
def compute_all_metrics(result):
    """Compute comprehensive metrics."""
    returns = result['return']
    
    total_return = result['cumulative'].iloc[-1] - 1
    annual_return = (1 + total_return) ** (52 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(52)
    sharpe = annual_return / volatility if volatility > 0 else 0
    
    cumulative = result['cumulative']
    drawdown = (cumulative / cumulative.cummax() - 1).min()
    
    avg_turnover = result['turnover'].mean()
    
    return {
        'Total Return': f"{total_return:.2%}",
        'Annual Return': f"{annual_return:.2%}",
        'Volatility': f"{volatility:.2%}",
        'Sharpe Ratio': f"{sharpe:.2f}",
        'Max Drawdown': f"{drawdown:.2%}",
        'Avg Turnover': f"{avg_turnover:.3f}"
    }

# Compute for all
comparison = pd.DataFrame({
    name: compute_all_metrics(result)
    for name, result in results_all.items()
})

print("\n" + "="*80)
print("REWARD FUNCTION COMPARISON")
print("="*80)
print(comparison)
print("="*80)
print("\nKey Takeaway: Your choice of reward function determines:")
print("  - Which commodities get selected")
print("  - How often you rebalance")
print("  - Your risk/return profile")
print("  - Transaction costs")
print("\nThere is no 'best' reward. It depends on your goals.")

## Step 9: Design Your Own Reward Function

Now it's your turn. Create a custom reward that matches YOUR trading goals.

In [ ]:
def custom_reward(arm_returns, all_returns, volatility=None, **kwargs):
    """
    Design your own reward function here.
    
    Example goal: Maximize returns while:
    - Keeping volatility under control
    - Avoiding large drawdowns
    - Minimizing regret
    
    Modify this function to match YOUR goals!
    """
    # Example: Combine multiple objectives
    
    # 1. Base return
    base_reward = arm_returns
    
    # 2. Penalize volatility
    if volatility is not None:
        vol_penalty = 0.5 * volatility  # Tune this weight
        base_reward = base_reward - vol_penalty
    
    # 3. Penalize regret
    best_return = all_returns.max()
    regret = best_return - arm_returns
    regret_penalty = 0.3 * regret  # Tune this weight
    
    final_reward = base_reward - regret_penalty
    
    return final_reward

# Test your custom reward
print("Testing custom reward function...\n")
custom_result = backtest_with_reward(weekly_returns, custom_reward, 'Custom')
custom_metrics = compute_all_metrics(custom_result)

print("Custom Reward Performance:")
for metric, value in custom_metrics.items():
    print(f"  {metric}: {value}")

print("\nModify the custom_reward function above and rerun!")
print("Try different penalty weights and see how it changes behavior.")

## Summary

**What you learned:**

1. **Raw returns are dangerous** → Create trend-chasing machines that buy tops
2. **Risk-adjusted rewards** → Conservative allocators that avoid volatility
3. **Regret-relative rewards** → Balanced exploration and exploitation
4. **Stability-weighted rewards** → Low turnover, tax/cost efficient

**The Critical Insight:**

> Your reward function IS your trading strategy.
>
> Choose it to match your real goals, not what sounds good.

**Practical Guidance:**

| Your Goal | Use This Reward |
|-----------|----------------|
| Steady accumulation | Stability-weighted |
| Beat benchmark | Regret-relative |
| Minimize risk | Risk-adjusted |
| Custom objectives | Build your own (multi-objective) |

**What's next:**
- [Regime-Aware Bandit](03_regime_commodity_bandit.ipynb): Add market regime detection
- [Reward Design Guide](../guides/02_reward_design_commodities.md): Deep dive into reward engineering
- [Guardrails Guide](../guides/03_guardrails_and_safety.md): Add safety constraints

In [ ]:
key_takeaways(["Raw returns are dangerous", "Risk-adjusted rewards", "Regret-relative rewards", "Stability-weighted rewards"])